<a href="https://colab.research.google.com/github/memo124/Laboratorio-de-IA-tica-Sesgos-y-Explicabilidad/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# INSTALACIÓN DE LIBRERÍAS FALTANTES
# ==========================================
!pip install shap lime

In [ ]:
# ==========================================
# ETAPA 2: IMPORTACIÓN Y LECTURA DE DATOS
# ==========================================
# Instalación de librerías necesarias (descomentar si falta alguna)
# !pip install pandas numpy scikit-learn shap lime matplotlib

import pandas as pd
import numpy as np
import shap
import lime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# 1. Cargar el dataset directamente desde el CSV crudo
url = "https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv"
df = pd.read_csv(url)

# 2. Exploración inicial
display(df.head())
print("Datos faltantes por columna:\n", df.isnull().sum())

# 3. Eliminar Loan_ID ya que no aporta valor predictivo
df = df.drop('Loan_ID', axis=1)

In [ ]:
# ==========================================
# ETAPA 3: PREPARACIÓN DE DATOS
# ==========================================

# 1. Separar X e y. Convertimos 'Loan_Status' a números (1 = Aprobado, 0 = Denegado)
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status'].map({'Y': 1, 'N': 0})

# 2. División en entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Identificar columnas numéricas y categóricas
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object']).columns

# 4. Rellenar valores faltantes (Mediana para numéricas, Moda para categóricas)
X_train_num = X_train[num_cols].fillna(X_train[num_cols].median())
X_test_num = X_test[num_cols].fillna(X_train[num_cols].median()) # Se usa la mediana del train

X_train_cat = X_train[cat_cols].fillna(X_train[cat_cols].mode().iloc[0])
X_test_cat = X_test[cat_cols].fillna(X_train[cat_cols].mode().iloc[0])

# 5. Convertir texto a números (One-Hot Encoding usando pandas)
X_train_cat_encoded = pd.get_dummies(X_train_cat, drop_first=True)
X_test_cat_encoded = pd.get_dummies(X_test_cat, drop_first=True)

# Alinear columnas por si alguna categoría solo existe en train pero no en test
X_train_cat_encoded, X_test_cat_encoded = X_train_cat_encoded.align(
    X_test_cat_encoded, join='left', axis=1, fill_value=0
)

# 6. Unir todo en DataFrames limpios y listos para el modelo
X_train_prep = pd.concat([X_train_num, X_train_cat_encoded], axis=1)
X_test_prep = pd.concat([X_test_num, X_test_cat_encoded], axis=1)

print("Forma de X_train_prep:", X_train_prep.shape)
print("Forma de X_test_prep:", X_test_prep.shape)

In [ ]:
# ==========================================
# ETAPA 4: ENTRENAMIENTO DEL MODELO
# ==========================================

# 1. Entrenar Random Forest
# Usamos un random_state para que los resultados sean reproducibles para el resto del equipo
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_prep, y_train)

# 2. Predicciones
y_pred = rf_model.predict(X_test_prep)

# 3. Métricas
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"F1-Score: {f1:.4f}")

In [ ]:
# ==========================================
# ETAPA 5: EXPLICABILIDAD GLOBAL (SHAP)
# ==========================================

# 1. Inicializar TreeExplainer
explainer = shap.TreeExplainer(rf_model)

# 2. Calcular los valores SHAP para el conjunto de prueba
shap_values = explainer.shap_values(X_test_prep)

# En Random Forest binario, shap_values tiene dos posiciones: [0] para Denegado, [1] para Aprobado.
# Explicaremos la clase Aprobado (índice 1). Si usas una versión nueva de SHAP, podría ser un solo array.
if isinstance(shap_values, list):
    shap_values_aprobado = shap_values[1]
elif len(shap_values.shape) == 3:
    shap_values_aprobado = shap_values[:, :, 1]
else:
    shap_values_aprobado = shap_values

# 3. Gráfico Beeswarm (Summary Plot)
plt.figure(figsize=(10, 6))
plt.title("SHAP Summary Plot - Clase: Aprobado (1)")
shap.summary_plot(shap_values_aprobado, X_test_prep, show=False)
plt.show()